In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient III: how far may one update move?

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part B (policy gradient), session 3 of its 4 sessions**
    - session 1 gave the theorem; session 2 filled slot 1, the weight on each step (an advantage)
    - **the limitation we start from:** every episode is expensive, and the policy gradient uses a batch of episodes for *one* update and then throws it away; a large step on a noisy estimate can also ruin the policy
    - this session fills the second slot: *how far may one update move?*
- **The question:** how can we take several updates from the same batch, and keep each one small enough to trust?
- **By the end you can**
    - say why a batch of episodes goes stale after one update, and what the probability ratio corrects
    - state the surrogate objective and why its gradient at ratio $1$ is the policy gradient
    - read the clipped objective, and say which samples it stops pushing
    - read the PPO outline and say which model does what
    - say how a trust region and clipping limit the movement, and what a KL penalty adds (developed in Part C)


**Where this session sits in Part B**

<img src="images/partB_arc_b3.png" width=95%>


# The limitation: a batch goes stale after one update

**Why this section?** Session 2 made the weight good. The next cost is how much each batch of episodes buys us.

- the policy gradient is **on-policy**: the episodes must come from the policy being improved (the idea from sampling and Q-learning)
    - the expectation in the theorem is over $\tau \sim \pi_\theta$
- one update changes $\theta$, so the old episodes now came from a *different* policy, the **old policy** $\pi_{\theta_{\text{old}}}$
    - REINFORCE and actor-critic (session 2) sample, update once, and discard
- two costs follow
    - **waste:** generating episodes is the expensive part, especially for language models; one update per batch uses little of what we paid for
    - **risk:** a bigger step would use the batch better, but the gradient is a *noisy local estimate*; far from $\theta_{\text{old}}$ it may point the wrong way, and a policy that collapses produces worse data for the next update
- the fix has two parts
    - **correct** for the fact that the data came from the old policy: the probability ratio (next slide)
    - **limit** how far $\theta$ moves from $\theta_{\text{old}}$: the surrogate and its constraints


## Reusing old data: the probability ratio

**Why this section?** This is the on-policy versus off-policy thread, now for a policy gradient: data from one policy, an expectation under another.

- the **probability ratio** compares the new policy with the old one, for one state and action
    - $r_\tt(\theta) = \dfrac{\pi_\theta(\actseq_\tt \mid \stateseq_\tt)}{\pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)}$
    - $r > 1$: the action became more likely; $r < 1$: less likely; $r = 1$ at $\theta = \theta_{\text{old}}$
- as an **importance weight**: averaging $f(\actseq)$ over actions *sampled by the old policy* and multiplying by $r$ gives the average under the *new* policy
    - $\E_{\actseq \sim \pi_{\theta_{\text{old}}}} \bigl[\, r(\actseq)\, f(\actseq) \,\bigr] = \E_{\actseq \sim \pi_\theta} \bigl[\, f(\actseq) \,\bigr]$
- by numbers (figure, synthetic): three actions, old $(0.5, 0.3, 0.2)$, new $(0.6, 0.25, 0.15)$, advantages $(+1, -1, -1)$
    - average advantage under the old policy: $0.5 - 0.3 - 0.2 = 0$ (as an advantage must, under the policy that generated the data)
    - average of $r \cdot$ advantage under the old policy: $0.5 \cdot 1.2 - 0.3 \cdot 0.833 - 0.2 \cdot 0.75 = 0.20$, the average advantage under the *new* policy ($0.6 - 0.25 - 0.15$): the improvement, in these states
    - the identity needs the old policy to give positive probability to every action the new policy can take

<img src="images/b3_ratio_bars.png" width=80%>

- notation: this $r$ is the PPO paper's ratio, not a reward (written $\rew$ here); context separates them


## The surrogate objective

**Why this section?** An objective we can optimize more than once on the same batch, whose gradient is still the policy gradient where it matters.

- the **surrogate**: the batch average of ratio times advantage
    - $L_{\text{sur}}(\theta) = \E_\tt \bigl[\, r_\tt(\theta) \, \advseq_\tt \,\bigr]$
    - $\E_\tt$ is the average over the time steps of the batch; the advantage comes from session 2, computed with the old policy
- it is a **local model** of the improvement over the old policy
    - its gradient at $\theta = \theta_{\text{old}}$ is the policy gradient of session 2, in the form we use (no $\disc^\tt$ factor)
    - it matches the true improvement only to first order, so away from $\theta_{\text{old}}$ we do not trust it
- why the gradient matches: the log-derivative trick returns
    - $\nabla_\theta r_\tt(\theta) = r_\tt(\theta) \, \nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt)$, because $\pi_{\theta_{\text{old}}}$ does not depend on $\theta$
    - at $r_\tt = 1$ this is $\nabla_\theta \log \pi_\theta \; \advseq_\tt$, the term of the unified form
- two cautions (admitted, not fixed here)
    - the ratio corrects the choice of *action* in each visited state, not *which states are visited*; that is why we stay near $\theta_{\text{old}}$
    - the surrogate is *maximized*; code minimizes its negative (a sign convention)


## Keeping $\theta$ near $\theta_{\text{old}}$: three mechanisms

**Why this section?** The surrogate is trustworthy only near $\theta_{\text{old}}$. Methods differ in how they enforce that.

| Mechanism | How | Used by |
|:--|:--|:--|
| **trust region** | a hard bound on the KL divergence between old and new policy (a measure of how different two distributions are) | TRPO (this session) |
| **clipping** | the gradient of a sample vanishes once its ratio has left a range, in the direction its advantage favors | PPO (this session) |
| **KL penalty** | a penalty on drifting away from another policy: the old one, or a frozen reference model | some PPO variants; RLHF-style PPO and GRPO (session 4 and Part C) |

- the first two limit the movement *within an iteration* (relative to the policy that made the batch)
- the third can anchor to a different policy: a model we do not want to leave at all; it adds $-\beta \, \KL(\cdot)$ to the objective, with a weight $\beta$ (session 4 and Part C)
- $\KL(p \,\|\, q)$ is the average, over samples from $p$, of $\log p - \log q$: zero when the two agree, larger as they differ, not symmetric
- PPO is the practical, widely used choice: no second-order machinery (that is, no second derivatives of the objective)


# PPO: clip the ratio

**Why this section?** The clipped objective is the whole idea of PPO (proximal policy optimization): stop pushing a sample once its ratio has moved far enough.

- the **clipped surrogate** (maximized)

$$
L_{\text{clip}}(\theta) = \E_\tt \Bigl[\, \min{} \bigl( r_\tt(\theta)\, \advseq_\tt, \;\; \text{clip}\bigl(r_\tt(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \advseq_\tt \bigr) \,\Bigr]
$$

- $\text{clip}(x, a, b)$ limits $x$ to the interval $[a, b]$; $\epsilon_{\text{clip}}$ is the clip range, typically $0.2$
- the paper calls the range $\epsilon$; we add the subscript to avoid the exploration probability $\epsilon$ of $\epsilon$-greedy
- clipping does **not** bound the policy: other samples share the parameters and can still push a clipped sample's ratio further, so PPO has no guarantee
- clipping does **not** force $r$ to stay in the range; it removes the gradient of a sample once its ratio has moved *past* the range in the direction its advantage favors
- the `min` keeps the pessimistic one: it never rewards a sample for moving further than the range allows, but it still pushes back a ratio that moved the *wrong* way


**What clipping does**

<img src="images/ppo_clip.png" width=95%>

- with a positive advantage the objective stops rewarding an increase of $r$ beyond $1 + \epsilon_{\text{clip}}$; with a negative advantage, a decrease below $1 - \epsilon_{\text{clip}}$
- where the curve is flat, the gradient is zero: that sample no longer pushes the policy
- the sloped branches keep the gradient, so a mistake (a ratio that moved the wrong way) can be undone


**Four samples, by numbers** ($\epsilon_{\text{clip}} = 0.2$, so the range is $[0.8, 1.2]$)

| Advantage | Ratio $r$ | $r \, \advseq$ | clipped term | `min` | Gradient |
|:--|:--|:--|:--|:--|:--|
| $+2$ | $1.3$ | $2.6$ | $1.2 \cdot 2 = 2.4$ | $2.4$ (clipped) | **zero**: already pushed up enough |
| $+2$ | $0.7$ | $1.4$ | $0.8 \cdot 2 = 1.6$ | $1.4$ (unclipped) | **flows**: pushes the ratio back up |
| $-1$ | $1.3$ | $-1.3$ | $1.2 \cdot (-1) = -1.2$ | $-1.3$ (unclipped) | **flows**: pushes the ratio back down |
| $-1$ | $0.7$ | $-0.7$ | $0.8 \cdot (-1) = -0.8$ | $-0.8$ (clipped) | **zero**: already pushed down enough |

- samples in the first and fourth rows have moved far enough in the direction their advantage asks
- samples in the second and third rows moved the wrong way, so the gradient still acts
- inside the range $[0.8, 1.2]$ nothing is clipped and the surrogate is the plain ratio objective


## Iterations and epochs: reusing one batch

**Why this section?** Reuse is the goal of the whole session. The words must be exact, because supervised learning uses them differently.

- in supervised learning, an **epoch** is one pass over fixed examples
- here the examples are episodes made by the *current* policy, so there are two nested loops
    - an **iteration** generates a batch with the policy as it is at the start, which becomes the old policy $\pi_{\theta_{\text{old}}}$
    - inside it, **epochs**: several passes over that *fixed* batch, updating the actor (and the critic) each time
    - at the end of the iteration $\theta_{\text{old}}$ is set to the new $\theta$
- why this is safe enough
    - the first update is exactly on-policy: $r_\tt = 1$ and the gradient is the policy gradient (implementations split a batch into mini-batches, so later mini-batches already have $r \neq 1$)
    - later epochs are slightly off-policy; the ratio corrects them, and clipping stops the samples that have moved far
- the number of epochs $K$ (often $3$ to $10$) is the dial: more epochs use the batch better, but the samples drift further from the old policy


## PPO in one loop

**Why this section?** Everything so far as one algorithm. The comments say which model computes what.

- **models:** actor $\pi_\theta$, critic $\statevalfun_\phi$, old actor $\pi_{\text{old}}$ (a frozen snapshot of the actor)
- for each iteration
    - **rollout:** collect actions, states and rewards with the current actor; set $\pi_{\text{old}} \leftarrow \pi_\theta$ (it produced the batch and is not updated below)
    - **frozen for the whole iteration (no gradient):** $\log \pi_{\text{old}}(\actseq_\tt \mid \stateseq_\tt)$ of each sampled step; the values $\statevalfun_\phi(\stateseq_\tt)$; the advantages $\hat{\advseq}_\tt$ by GAE from the rewards and values (session 2, appendix); the critic's targets $\hat{\advseq}_\tt + \statevalfun_\phi(\stateseq_\tt)$ (a terminal state has value $0$)
    - for epoch $= 1, \ldots, K$ (full-batch passes here; implementations split the batch into mini-batches)
        - ratio $r_\tt = \exp\bigl(\log \pi_\theta(\actseq_\tt \mid \stateseq_\tt) - \log \pi_{\text{old}}(\actseq_\tt \mid \stateseq_\tt)\bigr)$ (same actions, current weights)
        - actor loss: minus the mean over $\tt$ of $\min{}\bigl(r_\tt \hat{\advseq}_\tt, \; \text{clip}(r_\tt, 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}}) \hat{\advseq}_\tt\bigr)$ (minus: we minimize)
        - critic loss: the mean over $\tt$ of the squared difference between $\statevalfun_\phi(\stateseq_\tt)$ and its target
        - update $\theta$ and $\phi$ to reduce the actor loss plus $c$ times the critic loss ($c$ weights the critic loss; practice often adds an *entropy bonus*, a term that rewards keeping the policy random, to preserve exploration)

- the actor is updated every epoch; $\pi_{\text{old}}$ changes only between iterations
- the advantage is computed once per iteration, with the old critic, and held fixed
- *RLHF-style PPO adds a reference model and a KL term to this loop; that belongs to Part C*


**Which model does what**

| Model | Role | Updated | Needed for |
|:--|:--|:--|:--|
| actor $\pi_\theta$ | the policy we are improving | every epoch | the ratio's numerator, and the next iteration's data |
| old actor $\pi_{\text{old}}$ | snapshot that made the batch | between iterations | the ratio's denominator ($\log \pi_{\text{old}}$ stored once) |
| critic $\statevalfun_\phi$ | baseline source (slot 3) and the advantage via GAE | every epoch, by squared error | the advantage estimate |

- in practice the old actor is often only its stored log-probabilities, not a second network held in memory
- the critic is a **second set of parameters** to train; it may share layers with the actor, but for large language models it is typically a separate network of similar size; that cost matters in the next session


## Where the idea comes from: TRPO

**Why this section?** Clipping is a practical stand-in for a more principled constraint. The history explains why it looks the way it does.

- **TRPO** (trust region policy optimization): maximize the surrogate *subject to* a bound on how far the new policy may move
    - $\displaystyle \max{\theta} \;\E_\tt \bigl[ r_\tt(\theta)\, \advseq_\tt \bigr] \quad \text{subject to} \quad \E_{\state} \bigl[ \KL \bigl( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \bigr) \bigr] \leq \delta_{\text{KL}}$
    - the **trust region**: the set of policies we trust the local surrogate on
- TRPO is motivated by a monotonic-improvement bound for a KL-penalized version of the objective; the practical algorithm (an average-KL constraint with a chosen $\delta_{\text{KL}}$) approximates it and carries no such guarantee
    - the constraint needs second-order machinery, which is expensive for a large network
- **PPO** is a heuristic in the spirit of TRPO
    - clipping the ratio replaces the hard constraint, with no guarantee: the ratio can still leave the range
    - it uses plain first-order gradient steps, so it scales to large networks


## PPO on the threads, and the slot table

**Why this section?** Placing PPO in the grid we have been filling, so the next method is a single change.

- **on-policy versus off-policy:** each iteration is on-policy; the later epochs are slightly off-policy, corrected per action by the ratio (not for the state distribution)
- **bias and variance:** the weight is still the session-2 advantage; the critic lowers variance at the price of **bias**, and clipping buys stability at the price of some bias (it truncates the gradient)
- **evaluate versus improve:** the critic evaluates and the actor improves, the same loop as in value-based control
- **backup depth:** GAE sets how far the advantage looks ahead (appendix to session 2)
- **slots** (the plan from session 1): slot 1 is the weight (session 2), slot 2 is how far one update moves (this session), slot 3 is where the baseline comes from, inside slot 1 (sessions 2 and 4)

| Method | Slot 1: weight $\advseq$ | Slot 2: how far | Slot 3: baseline source |
|:--|:--|:--|:--|
| raw theorem | return, no baseline | plain step | none |
| REINFORCE | return minus baseline | plain step | moving average |
| actor-critic | TD error or GAE | plain step | learned critic |
| **PPO** | GAE with a learned critic | **clipped ratio, $K$ epochs per batch** | learned critic |
| GRPO | group-relative advantage | clipped ratio, as PPO (session 4) | mean of a group of episodes |


## The new limitation: the critic

**Why this section?** PPO's weight comes from a critic. That is a cost of its own.

- PPO needs a learned critic
    - a second network to train, store and run (for large language models, often as large as the policy)
    - the critic's mistakes become bias in the advantage
    - for a language model with a reward only at the end of a long response, the critic must score every prefix
- the question for the next session: **do we need the critic at all?**
    - sample *several* episodes from the same start and use their *own average* as the baseline
    - slot 3 changes; slot 2 keeps the clipped ratio


## Aside: the Kullback-Leibler direction in TRPO

- $\KL(p \,\|\, q) = \E_{x \sim p}\bigl[ \log p(x) - \log q(x) \bigr]$ is not symmetric in $p$ and $q$
- TRPO writes $\pi_{\theta_{\text{old}}}$ first: the expectation is over the *old* policy, the one that generated the data
- an action the new policy introduces but the old one never took has weight $0$ in this expectation, so it has no *direct* effect on the constraint, only an indirect one through the probability it takes from other actions


# What we established, and what it sets up

- **Established**
    - the policy gradient is on-policy, so a batch goes stale after one update; reusing it needs a correction and a limit
    - the **probability ratio** is an importance weight: old-policy samples, new-policy average
    - the **surrogate** $\E[r_\tt \advseq_\tt]$ has the policy gradient as its gradient at ratio $1$
    - **PPO** clips the ratio, so samples that have moved far enough in the direction of their advantage stop pushing; this allows several epochs per batch
    - **TRPO** pursues the same goal as clipping with a hard KL constraint; the KL penalty (to a frozen reference) comes with language models
    - PPO's remaining cost: a critic, a second large network
- **Sets up**
    - next session: slot 3, do we need the critic? GRPO uses a group of episodes as the baseline and keeps the clipped ratio
